## Hybrid KG QA (GraphDB + Chroma RAG)

Routes each question through an LLM **route planner**, then either:

1. **Chroma** — verbalized entity-centric chunks (`kg_entities_ontology_sliced` in `chroma_db/`)
2. **GraphDB** — ontology-guided SPARQL against repository `Master_Thesis`

In [1]:
# Install once if needed
!pip install -q chromadb requests openai rdflib

In [2]:
from pathlib import Path
import json
import os
import re
import hashlib
from typing import Any, Dict, List, Optional

import requests
import chromadb
from openai import OpenAI
from rdflib import Graph, RDF, RDFS, OWL, URIRef

project_root = Path('.').resolve()
ontology_path = project_root / 'ontology.ttl'

# ---- GraphDB ----
GRAPHDB_BASE = 'http://localhost:7200'
REPOSITORY_ID = 'Master_Thesis'
SPARQL_ENDPOINT = f"{GRAPHDB_BASE}/repositories/{REPOSITORY_ID}"

# ---- LLM ----
LLM_BASE_URL = os.getenv('LLM_BASE_URL', 'https://web.ollama-gpt-oss.ai.wu.ac.at/api')
LLM_API_KEY = os.getenv('LLM_API_KEY', 'sk-9b41b856cc0b403b8a3c10618f1c2996')
LLM_MODEL = 'gpt-oss:120b'

llm_client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    timeout=120.0,
    max_retries=4,
)

# ---- Chroma (attach only) ----
OLLAMA_EMBED_BASE = os.getenv('OLLAMA_EMBED_BASE', 'http://localhost:11434')
EMBED_MODEL = os.getenv('EMBED_MODEL', 'bge-m3')
COLLECTION_NAME = 'kg_entities_ontology_sliced'
CHROMA_PATH = project_root / 'chroma_db'

print('Project root:', project_root)
print('Ontology:', ontology_path.exists())
print('GraphDB:', SPARQL_ENDPOINT)
print('LLM:', LLM_MODEL)
print('Chroma path:', CHROMA_PATH)


Project root: C:\Users\dyury\Desktop\Master Thesis
Ontology: True
GraphDB: http://localhost:7200/repositories/Master_Thesis
LLM: gpt-oss:120b
Chroma path: C:\Users\dyury\Desktop\Master Thesis\chroma_db


In [3]:
def call_llm(prompt: str, model: str = LLM_MODEL) -> str:
    resp = llm_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
    )
    return (resp.choices[0].message.content or '').strip()


In [4]:
_CORE_NS = 'https://w3id.org/football-cdf/core#'

#  collapses long absolute URIs into the prefixed forms the LLM is expected to write in SPARQL
def _short(uri: str) -> str:
    if uri.startswith(_CORE_NS):
        return 'core:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/2001/XMLSchema#'):
        return 'xsd:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/2000/01/rdf-schema#'):
        return 'rdfs:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/1999/02/22-rdf-syntax-ns#'):
        return 'rdf:' + uri.split('#', 1)[1]
    if uri.startswith('http://www.w3.org/2002/07/owl#'):
        return 'owl:' + uri.split('#', 1)[1]
    return f'<{uri}>'

# Return short names for an rdfs:domain/range value, expanding owl:unionOf lists.
def _expand_domain_or_range(g: Graph, node) -> list[str]:
    out: list[str] = []
    if isinstance(node, URIRef):
        out.append(_short(str(node)))
        return out
    for u in g.objects(node, OWL.unionOf):
        cur = u
        while cur is not None and cur != RDF.nil:
            for first in g.objects(cur, RDF.first):
                if isinstance(first, URIRef):
                    out.append(_short(str(first)))
            rests = list(g.objects(cur, RDF.rest))
            cur = rests[0] if rests else None
    return out

# Loads ontology.ttl into an in-memory rdflib graph.
# Build a richer schema hint for SPARQL generation.
# Includes classes plus object/datatype properties with their declared
# rdfs:domain, rdfs:range, and short rdfs:comment when available.
# max_items=0 means no truncation; positive values cap each section.
def load_ontology_schema_for_prompt(path: Path, max_items: int = 0) -> str:
    g = Graph()
    g.parse(path, format='turtle')

    classes: list[str] = []
    for s in sorted(set(g.subjects(RDF.type, OWL.Class)), key=str):
        if isinstance(s, URIRef) and str(s).startswith(_CORE_NS):
            classes.append(_short(str(s)))

# this allows model to see exactly which subject a property is allowed on, which prevents misuse -> core:<name> | domain: core:<X> | range: <Y> -- <short comment>
    def gather_props(p_type) -> list[str]:
        rows: list[str] = []
        for p in sorted(set(g.subjects(RDF.type, p_type)), key=str):
            if not isinstance(p, URIRef):
                continue
            p_uri = str(p)
            if not p_uri.startswith(_CORE_NS):
                continue
            doms: list[str] = []
            for d in g.objects(p, RDFS.domain):
                doms.extend(_expand_domain_or_range(g, d))
            rngs: list[str] = []
            for r in g.objects(p, RDFS.range):
                rngs.extend(_expand_domain_or_range(g, r))
            doms = sorted(set(doms))
            rngs = sorted(set(rngs))
            comment = next((str(c) for c in g.objects(p, RDFS.comment)), '').strip().replace('\n', ' ')
            if len(comment) > 90:
                comment = comment[:87] + '...'
            parts = [_short(p_uri)]
            if doms:
                parts.append(f"domain: {', '.join(doms)}")
            if rngs:
                parts.append(f"range: {', '.join(rngs)}")
            line = ' | '.join(parts)
            if comment:
                line += f' -- {comment}'
            rows.append(line)
        return rows

    obj_props = gather_props(OWL.ObjectProperty)
    data_props = gather_props(OWL.DatatypeProperty)

    if max_items and max_items > 0:
        classes = classes[:max_items]
        obj_props = obj_props[:max_items]
        data_props = data_props[:max_items]

    sections = []
    if classes:
        sections.append('Classes:\n' + '\n'.join('- ' + c for c in classes))
    if obj_props:
        sections.append('Object properties:\n' + '\n'.join('- ' + p for p in obj_props))
    if data_props:
        sections.append('Datatype properties:\n' + '\n'.join('- ' + p for p in data_props))
    return '\n\n'.join(sections)


DATA_GRAPH_HINT = """
Instance data (fcdf_kg.ttl) often uses:
- core:events_goals, core:events_cards, core:events_subtitutions (subproperties of core:events from Match to Goal / Card / Subtitution)
- core:related_event_ids (links between Event resources)
- core:pass_outcome_type, core:pass_type (Passes); core:shot_outcome_type, core:shot_type (Shots / Goals)
""".strip()

schema_hint = load_ontology_schema_for_prompt(ontology_path) + "\n\n" + DATA_GRAPH_HINT
print('\n'.join(schema_hint.splitlines()[:20]))
if len(schema_hint.splitlines()) > 20:
    print('...')

Classes:
- core:Card
- core:Competition
- core:Event
- core:Goal
- core:Match
- core:Match_Result
- core:Match_Status
- core:Meta
- core:Misc
- core:Pass
- core:Player
- core:Referee
- core:Season
- core:Shot
- core:Subtitution
- core:Team
- core:Vendor
- core:Whistle

...


In [5]:
def extract_sparql(text: str) -> str:
    # Prefer fenced ```sparql blocks if present.
    match = re.search(r"```(?:sparql)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text.strip()


try:
    from rdflib.plugins.sparql.parser import parseQuery
except ImportError:
    parseQuery = None  # type: ignore[misc, assignment]


# Rewrite invalid 4-token triples ?S core:P rdfs:label|comment ?O into one property path.
def fix_invalid_core_then_rdfs_label(q: str) -> str:
    for rdfp in ("rdfs:label", "rdfs:comment"):
        q = re.sub(
            rf"(?P<subj>\?\w+)\s+(?P<core>core:[A-Za-z_][A-Za-z0-9_]*)\s+{rdfp}\s+(?P<obj>\?\w+)\s*\.",
            rf"\g<subj> \g<core>/{rdfp} \g<obj> .",
            q,
        )
        q = re.sub(
            rf"(?P<subj>\?\w+)\s+(?P<core>core:[A-Za-z_][A-Za-z0-9_]*)\s+{rdfp}\s+(?P<lit>\"(?:[^\"\\\\]|\\\\.)*\")\s*\.",
            rf"\g<subj> \g<core>/{rdfp} \g<lit> .",
            q,
        )
    return q


def normalize_sparql(query: str) -> str:
    q = query.strip()

    # Fix malformed 'PREFIX:' pattern into a real prefix declaration.
    q = re.sub(
        r"(?im)^\s*PREFIX\s*:\s*<https?://w3id\.org/football-cdf/core#>\s*$",
        "PREFIX core: <https://w3id.org/football-cdf/core#>",
        q,
    )

    # Ensure prefix declarations exist when prefixed names are used.
    if re.search(r"\bcore:[A-Za-z_]", q) and 'PREFIX core:' not in q:
        q = "PREFIX core: <https://w3id.org/football-cdf/core#>\n" + q

    if re.search(r"\bxsd:[A-Za-z_]", q) and 'PREFIX xsd:' not in q:
        q = "PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\n" + q

    if re.search(r"\brdfs:[A-Za-z_]", q) and 'PREFIX rdfs:' not in q:
        q = "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n" + q

    # Align with existing data where ids are typically string literals.
    q = re.sub(r'"(\d+)"\^\^xsd:integer', r'"\1"', q)

    # Fix common bad UNION layout: WHERE { block1 } UNION { block2 }
    union_outside_where = re.search(
        r"(?is)(.*?SELECT\s+.*?WHERE\s*)\{\s*(.*?)\s*\}\s*UNION\s*\{\s*(.*?)\s*\}\s*$",
        q,
    )
    if union_outside_where:
        prefix_select = union_outside_where.group(1).strip()
        block_1 = union_outside_where.group(2).strip()
        block_2 = union_outside_where.group(3).strip()
        q = (
            f"{prefix_select}{{\n"
            f"  {{\n{block_1}\n  }}\n"
            f"  UNION\n"
            f"  {{\n{block_2}\n  }}\n"
            f"}}"
        )

    q = fix_invalid_core_then_rdfs_label(q)
    return q


_ALLOWED_CORE_TERMS_CACHE = None


# Build an allowlist of football-cdf core terms from ontology.ttl.
# Cached so we do not re-parse ontology on every question.
def get_allowed_core_terms() -> set[str]:
    global _ALLOWED_CORE_TERMS_CACHE
    if _ALLOWED_CORE_TERMS_CACHE is not None:
        return _ALLOWED_CORE_TERMS_CACHE

    g = Graph()
    g.parse(ontology_path, format='turtle')

    q = """
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX owl: <http://www.w3.org/2002/07/owl#>

    SELECT ?term WHERE {
      {
        ?term rdf:type owl:Class .
      }
      UNION
      {
        ?term rdf:type owl:ObjectProperty .
      }
      UNION
      {
        ?term rdf:type owl:DatatypeProperty .
      }
    }
    """

    allowed = set()
    for row in g.query(q):
        term = str(row.term)
        if term.startswith('https://w3id.org/football-cdf/core#'):
            allowed.add(term.split('#', 1)[1])

    _ALLOWED_CORE_TERMS_CACHE = allowed
    return allowed


def validate_core_terms(query: str) -> tuple[bool, str]:
    allowed = get_allowed_core_terms()
    used = set(re.findall(r'\bcore:([A-Za-z_][A-Za-z0-9_]*)\b', query))
    unknown = sorted(t for t in used if t not in allowed)
    if unknown:
        preview = ', '.join(unknown[:8])
        if len(unknown) > 8:
            preview += ', ...'
        return False, f'Unknown core: terms not present in ontology: {preview}'
    return True, 'ok'


_PROP_DOMAIN_CACHE: dict[str, set[str]] | None = None


# Map core property local-name -> set of declared domain class local-names.
# Properties without rdfs:domain are absent from the map (treated as universal).
# Handles owl:unionOf domain declarations.
def _load_property_domains() -> dict[str, set[str]]:
    global _PROP_DOMAIN_CACHE
    if _PROP_DOMAIN_CACHE is not None:
        return _PROP_DOMAIN_CACHE
    g = Graph()
    g.parse(ontology_path, format='turtle')
    domains: dict[str, set[str]] = {}
    for p, _, d in g.triples((None, RDFS.domain, None)):
        if not isinstance(p, URIRef) or not str(p).startswith(_CORE_NS):
            continue
        p_local = str(p).split('#', 1)[1]
        classes: list[str] = []
        if isinstance(d, URIRef):
            if str(d).startswith(_CORE_NS):
                classes.append(str(d).split('#', 1)[1])
        else:
            for u in g.objects(d, OWL.unionOf):
                cur = u
                while cur is not None and cur != RDF.nil:
                    for first in g.objects(cur, RDF.first):
                        if isinstance(first, URIRef) and str(first).startswith(_CORE_NS):
                            classes.append(str(first).split('#', 1)[1])
                    rests = list(g.objects(cur, RDF.rest))
                    cur = rests[0] if rests else None
        if classes:
            domains.setdefault(p_local, set()).update(classes)
    _PROP_DOMAIN_CACHE = domains
    return _PROP_DOMAIN_CACHE


def validate_property_domains(query: str) -> tuple[bool, str]:
    domains = _load_property_domains()
    var_class: dict[str, set[str]] = {}
    for m in re.finditer(
        r'\?([A-Za-z_][A-Za-z0-9_]*)\s+(?:a|rdf:type)\s+core:([A-Za-z_][A-Za-z0-9_]*)',
        query,
    ):
        var_class.setdefault(m.group(1), set()).add(m.group(2))

    violations: list[str] = []
    for m in re.finditer(
        r'\?([A-Za-z_][A-Za-z0-9_]*)\s+core:([A-Za-z_][A-Za-z0-9_]*)\b',
        query,
    ):
        var, prop = m.group(1), m.group(2)
        if var not in var_class:
            continue
        prop_domains = domains.get(prop)
        if not prop_domains:
            continue
        if not (var_class[var] & prop_domains):
            violations.append(
                f"core:{prop} (domain: {', '.join(sorted(prop_domains))}) used on "
                f"?{var} (typed as core:{', core:'.join(sorted(var_class[var]))})"
            )

    if violations:
        violations = sorted(set(violations))
        return False, 'Property used on wrong subject class: ' + '; '.join(violations[:4])
    return True, 'ok'


def is_likely_valid_sparql(query: str) -> tuple[bool, str]:
    q = query.strip()
    q_upper = q.upper()

    if not q:
        return False, 'Empty query.'
    if 'SELECT' not in q_upper:
        return False, 'Only SELECT queries are supported in this notebook.'
    if q.count('{') != q.count('}'):
        return False, 'Unbalanced braces in query.'
    if re.search(r'(?im)^\s*PREFIX\s*:', q):
        return False, 'Malformed PREFIX declaration. Use e.g. PREFIX core: <...>.'

    # Heuristic: UNION must be followed by a grouped block (after optional whitespace).
    if re.search(r'\bUNION\b(?!\s*\{)', q_upper):
        return False, 'UNION must be followed by a grouped block: UNION { ... }'

    ok_terms, reason_terms = validate_core_terms(q)
    if not ok_terms:
        return False, reason_terms

    ok_dom, reason_dom = validate_property_domains(q)
    if not ok_dom:
        return False, reason_dom

    if parseQuery is not None:
        try:
            parseQuery(q)
        except Exception as e:
            msg = str(e).replace("\n", " ")
            if len(msg) > 220:
                msg = msg[:220] + "..."
            return False, f"SPARQL parse error: {msg}"

    return True, 'ok'


def generate_sparql(question: str, schema: str) -> str:
    prompt = f"""
You create SPARQL SELECT queries for GraphDB.

Use only terms from this ontology summary:
{schema}

Rules:
- Return exactly ONE executable SPARQL SELECT query.
- Use explicit prefixes when needed, for example:
  PREFIX core: <https://w3id.org/football-cdf/core#>
  PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
- Never output malformed PREFIX syntax like 'PREFIX: <...>'.
- Keep query syntax strict and valid for GraphDB.
- Respect each property's declared `domain:` from the schema above. Never use
  a property on a subject of an unrelated class (e.g. core:has_played has
  domain core:Player, so it must NOT appear on a ?match typed as core:Match).
- Each basic triple pattern is exactly THREE terms then a dot: subject predicate object .
  NEVER write four terms on one line (e.g. ?m core:teams_home rdfs:label ?n is INVALID).
- To read a label after an object property (team, competition, season, referee), use either TWO triples:
  ?m core:teams_home ?t . ?t rdfs:label ?name .
  OR one property path: ?m core:teams_home/rdfs:label ?name .
  Same idea for core:teams_away, core:competition, core:season, core:referee.
- If you use UNION, each branch MUST be wrapped in braces INSIDE one outer WHERE block:
  WHERE {{ {{ ... }} UNION {{ ... }} }}
- For team names, prefer rdfs:label on the Team resource (not core:name on the match).
- Do not output any explanation or markdown, only the query.
- If unsure, still produce a best-effort valid SELECT query.

User question:
{question}
    """.strip()

    raw = call_llm(prompt)
    return normalize_sparql(extract_sparql(raw))


def repair_sparql(question: str, bad_query: str, error_hint: str, schema: str) -> str:
    prompt = f"""
Fix this SPARQL query so it is valid and executable in GraphDB.

Question:
{question}

Ontology summary:
{schema}

Validation error:
{error_hint}

Bad query:
{bad_query}

Rules:
- Return exactly one corrected SPARQL SELECT query.
- Keep the intent of the original question.
- Each triple pattern must be subject predicate object only (three terms before the dot).
  Never chain rdfs:label as a second predicate on the same line as core:teams_home (use ?m core:teams_home/rdfs:label ?n or two triples).
- If UNION is used, format as {{ ... }} UNION {{ ... }}.
- Use proper prefix declarations (e.g., PREFIX core: <...>).
- Output query only, no explanation.
    """.strip()

    return normalize_sparql(extract_sparql(call_llm(prompt)))


def run_sparql(query: str, endpoint: str = SPARQL_ENDPOINT) -> dict:
    headers = {'Accept': 'application/sparql-results+json'}
    resp = requests.get(endpoint, params={'query': query}, headers=headers, timeout=720)
    resp.raise_for_status()
    return resp.json()


def run_sparql_safe(
    question: str,
    query: str,
    schema: str,
    endpoint: str = SPARQL_ENDPOINT,
    max_repair_attempts: int = 2,
) -> tuple[str, dict]:
    """
    Robust generic loop:
    - normalize
    - validate syntax + ontology terms
    - attempt execution
    - if failing, repair with explicit error feedback and retry
    """
    current_query = normalize_sparql(query)
    last_error = None

    for _ in range(max_repair_attempts + 1):
        ok, reason = is_likely_valid_sparql(current_query)
        if not ok:
            last_error = f'Validation failed: {reason}'
            current_query = normalize_sparql(repair_sparql(question, current_query, last_error, schema))
            continue

        try:
            result_json = run_sparql(current_query, endpoint=endpoint)
            bindings = result_json.get('results', {}).get('bindings', [])
            if bindings:
                return current_query, result_json

            # Query was valid but likely too restrictive/wrong relation: repair once using result feedback.
            last_error = 'Query executed but returned no rows. Try a semantically close alternative relation from ontology.'
            current_query = normalize_sparql(repair_sparql(question, current_query, last_error, schema))
        except requests.HTTPError as e:
            last_error = f'GraphDB HTTP error: {e}'
            current_query = normalize_sparql(repair_sparql(question, current_query, last_error, schema))

    raise RuntimeError(f'Unable to produce executable SPARQL after retries. Last error: {last_error}')


def format_bindings(result_json: dict, max_rows: int = 30) -> str:
    head_vars = result_json.get('head', {}).get('vars', [])
    bindings = result_json.get('results', {}).get('bindings', [])
    if not bindings:
        return 'No rows returned.'

    lines = []
    for i, row in enumerate(bindings[:max_rows], start=1):
        vals = []
        for v in head_vars:
            vals.append(f"{v}={row.get(v, {}).get('value', '')}")
        lines.append(f"{i}. " + '; '.join(vals))
    if len(bindings) > max_rows:
        lines.append(f"... ({len(bindings) - max_rows} more rows)")
    return '\n'.join(lines)


def answer_from_results(question: str, query: str, result_json: dict) -> str:
    table_text = format_bindings(result_json)
    prompt = f"""
You are answering a user question using only GraphDB query results.

Question:
{question}

SPARQL query used:
{query}

Query results:
{table_text}

Rules:
- Use only the provided results.
- If results are empty or insufficient, say you do not know.
- Be concise.
    """.strip()

    return call_llm(prompt)


def graphdb_qa(question: str) -> dict:
    initial_sparql = generate_sparql(question, schema_hint)
    final_sparql, result_json = run_sparql_safe(question, initial_sparql, schema_hint)
    answer = answer_from_results(question, final_sparql, result_json)
    return {
        'question': question,
        'initial_sparql': initial_sparql,
        'sparql': final_sparql,
        'result_json': result_json,
        'answer': answer,
    }

In [6]:
class OllamaEmbeddingFunction:
    def __init__(self, model: str = EMBED_MODEL, base_url: str = OLLAMA_EMBED_BASE):
        self.model = model
        self.url = f'{base_url}/api/embeddings'

    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        vectors: List[List[float]] = []
        for text in texts:
            resp = requests.post(
                self.url, json={'model': self.model, 'prompt': text}, timeout=120,
            )
            resp.raise_for_status()
            vectors.append(resp.json()['embedding'])
        return vectors

    def embed_documents(self, input: List[str]) -> List[List[float]]:
        return self._embed_texts(input)

    def embed_query(self, input):
        if isinstance(input, str):
            return self._embed_texts([input])[0]
        return self._embed_texts(input)

    def __call__(self, input: List[str]) -> List[List[float]]:
        return self.embed_documents(input)

    def name(self) -> str:
        h = hashlib.sha1(self.model.encode('utf-8')).hexdigest()[:8]
        return f'ollama-{self.model.replace(":", "-")}-{h}'


chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
embedding_function = OllamaEmbeddingFunction(model=EMBED_MODEL, base_url=OLLAMA_EMBED_BASE)
collection = chroma_client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
)
print('Chroma collection:', COLLECTION_NAME, '| count:', collection.count())


Chroma collection: kg_entities_ontology_sliced | count: 139731


In [7]:
MAX_PER_OWL_CLASS = 5
EXCLUDED_SLICE_PREFIXES = ('events_',)


def _filter_and_diversify_results(results: dict, top_k: int) -> dict:
    """Drop events_* slices and cap repeated owl_class."""
    ids = results.get('ids', [[]])[0]
    documents = results.get('documents', [[]])[0]
    metadatas = results.get('metadatas', [[]])[0]
    distances = results.get('distances', [[]])[0]

    class_counts: Dict[str, int] = {}
    out_ids, out_docs, out_meta, out_dist = [], [], [], []

    for doc_id, text, meta, dist in zip(ids, documents, metadatas, distances):
        meta = meta or {}
        sl = str(meta.get('slice', ''))
        if any(sl.startswith(p) for p in EXCLUDED_SLICE_PREFIXES):
            continue
        oc = str(meta.get('owl_class', ''))
        if oc:
            if class_counts.get(oc, 0) >= MAX_PER_OWL_CLASS:
                continue
            class_counts[oc] = class_counts.get(oc, 0) + 1
        out_ids.append(doc_id)
        out_docs.append(text)
        out_meta.append(meta)
        out_dist.append(dist)
        if len(out_ids) >= top_k:
            break

    return {
        'ids': [out_ids],
        'documents': [out_docs],
        'metadatas': [out_meta],
        'distances': [out_dist],
    }


def show_retrieved_chunks(results: dict, question: str | None = None) -> None:
    ids = results.get('ids', [[]])[0]
    documents = results.get('documents', [[]])[0]
    metadatas = results.get('metadatas', [[]])[0]
    distances = results.get('distances', [[]])[0]

    if question:
        print(f'Question: {question}\n')
    print(f'Retrieved {len(documents)} chunk(s):\n')
    print('=' * 80)

    for rank, (doc_id, text, meta) in enumerate(zip(ids, documents, metadatas), start=1):
        meta = meta or {}
        dist = distances[rank - 1] if rank - 1 < len(distances) else None
        dist_str = f'{dist:.4f}' if dist is not None else 'n/a'
        print(f'[{rank}] id={doc_id}  distance={dist_str}')
        print(
            f'    focal={meta.get("focal", "")}  slice={meta.get("slice", "")}  '
            f'owl_class={meta.get("owl_class", "")}  n_lines={meta.get("n_lines", "")}'
        )
        print('-' * 80)
        print(text)
        print('=' * 80)
        print()


def retrieve_chunks(question: str, top_k: int = 10) -> dict:
    raw = collection.query(query_texts=[question], n_results=max(top_k * 3, top_k + 5))
    return _filter_and_diversify_results(raw, top_k)


def chroma_qa(
    question: str,
    top_k: int = 10,
    show_retrieved: bool = False,
) -> dict:
    results = retrieve_chunks(question, top_k=top_k)

    if show_retrieved:
        show_retrieved_chunks(results, question=question)

    context = '\n\n'.join(results['documents'][0])
    prompt = f"""
You answer using ONLY the facts in the context blocks below.

Rules:
- Give a direct factual answer about the subject of the question.
- Do NOT describe or summarize "the context" — state what happened, who, when, where.
- Prefer Team and match_core blocks for team/match overviews; use event blocks for specific actions.
- Quote literal ids (event/..., match/..., player/..., team/...) verbatim when you cite them.
- If the answer is not supported, say you do not know.

Context:
{context}

Question: {question}
Answer:
""".strip()

    if show_retrieved:
        print('--- LLM answer ---\n')

    answer = call_llm(prompt)
    return {
        'question': question,
        'route': 'chroma',
        'retrieved_ids': results.get('ids', [[]])[0],
        'chunks': results.get('documents', [[]])[0],
        'metadatas': results.get('metadatas', [[]])[0],
        'answer': answer,
    }


In [8]:
def _parse_route_json(text: str) -> dict:
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and obj.get('route') in ('chroma', 'graphdb'):
            return {'route': obj['route'], 'reason': str(obj.get('reason', ''))}
    except json.JSONDecodeError:
        pass
    m = re.search(r'\{[^{}]*"route"\s*:\s*"(chroma|graphdb)"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            return {'route': obj['route'], 'reason': str(obj.get('reason', ''))}
        except json.JSONDecodeError:
            pass
    if 'graphdb' in text.lower():
        return {'route': 'graphdb', 'reason': 'fallback parse'}
    return {'route': 'chroma', 'reason': 'fallback parse'}


def plan_route(question: str) -> dict:
    prompt = f"""
You route football KG questions to one backend.

Backends:
- chroma: verbalized text chunks — good for narrative facts, single events/players,
  "what happened", descriptions, straightforward lookups from prose.
- graphdb: SPARQL over the full graph — good for counts, aggregates, multi-hop joins,
  listing all matches/players meeting criteria, comparisons, calculations,
  "how many", "list all", "every match where".

Reply with JSON only, no markdown:
{{"route": "chroma" or "graphdb", "reason": "one short sentence"}}

Question: {question}
""".strip()
    raw = call_llm(prompt)
    return _parse_route_json(raw)


def hybrid_qa(
    question: str,
    *,
    force_route: Optional[str] = None,
    show_trace: bool = True,
    show_retrieved: bool = False,
    top_k: int = 10,
) -> dict:
    if force_route:
        route_info = {'route': force_route, 'reason': 'forced'}
    else:
        route_info = plan_route(question)

    route = route_info['route']
    if show_trace:
        print(f'Route: {route} — {route_info.get("reason", "")}\n')

    if route == 'graphdb':
        out = graphdb_qa(question)
        out['route'] = 'graphdb'
        out['route_reason'] = route_info.get('reason', '')
        if show_trace:
            print('SPARQL:\n', out.get('sparql', ''))
            n = len(out.get('result_json', {}).get('results', {}).get('bindings', []))
            print(f'Rows: {n}\n')
            print('--- LLM answer ---\n')
            print(out.get('answer', ''))
        return out

    out = chroma_qa(question, top_k=top_k, show_retrieved=show_retrieved)
    out['route_reason'] = route_info.get('reason', '')
    if show_trace and not show_retrieved:
        ids = out.get('retrieved_ids', [])
        print(f'Retrieved {len(ids)} chunks')
        for i, doc_id in enumerate(ids[:5], 1):
            print(f'  [{i}] {doc_id}')
        if len(ids) > 5:
            print(f'  ... ({len(ids) - 5} more)')
        print()
        print('--- LLM answer ---\n')
        print(out.get('answer', ''))
    return out


In [9]:
question = "Which teams played in match 3895052?"
result = hybrid_qa(question, show_trace=True)

Route: chroma — simple factual lookup of teams in a specific match

Retrieved 0 chunks

--- LLM answer ---

I do not know.


In [10]:
question = "What can you say about RB Leipzig?"
result = hybrid_qa(question, show_trace=True, top_k=15)

Route: chroma — seeks a narrative description of the club

Retrieved 5 chunks
  [1] event/0099a66e-fedc-4420-8775-364add41fcca::part::0
  [2] event/ab2ebb82-5b19-4436-a165-db0ccf4b17c7::part::0
  [3] event/1704aeec-c254-4901-9658-3dd27b186479::part::0
  [4] event/81a2c2e2-73e9-40ca-92e0-f79140a47028::part::0
  [5] event/f4ffc7c3-d294-4904-b27a-4a7788f363e5::part::0

--- LLM answer ---

- In match **match/3895202**, RB Leipzig defender **Lukas Klostermann** made a right‑foot pass in the first half at **00:48:00.584** from the position **(7.1, 70.9)** toward **(43.7, 64.5)**; the ball was received by **Benjamin Šeško** (event/f6e9131b-fa48-4172-8824-fe3c629db878).  

- In the same match (**match/3895202**), **Lukas Klostermann** made another right‑foot pass in the second half at **00:53:51.169** from **(77.9, 36.9)** toward **(83.5, 51.7)**; the ball was received by **Daniel Olmo Carvajal** (event/f44870ad-1133-4331-a5a3-a079bdcb3263).  

- In match **match/3895052**, RB Leipzig midfield

In [11]:
question = "How many goals were scored in match 3895052?"
result = hybrid_qa(question, show_trace=True)

Route: graphdb — needs an aggregate count for a specific match

SPARQL:
 PREFIX core: <https://w3id.org/football-cdf/core#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT (COUNT(?g) AS ?goalCount)
WHERE {
  ?match a core:Match .
  ?match core:id "3895052" .
  ?match core:events_goals ?g .
  ?g a core:Goal .
}
Rows: 1

--- LLM answer ---

The match with ID 3895052 had **5 goals** scored.
